# 01: Build and Annotate the 260-Image Benchmark

This notebook creates a corruption-checked, deduplicated, class-balanced benchmark. It seals train, validation, and test membership before preparing two independent visible-ingredient annotation passes.

Primary task: **per-image visible-component recognition**. Recipe knowledge, hidden ingredients, web associations, food naming, and nutrition are not ground truth for this task.

Run this notebook first. Notebook 02 is intentionally blocked until both annotation passes and human adjudication are complete.

## Define the target and frozen ontology

The ontology uses canonical multiword component labels and gives every label a visual-evidence rule. Recipe-writing fragments and preparation terms such as `all`, `at`, `freshly`, and `coarsely` are not model targets.

This keeps the task visually defensible. Hidden recipe ingredients cannot be scored as observable image evidence. Label validity therefore takes priority over vocabulary breadth, and hidden spices such as garlic and turmeric are outside the primary ground truth unless they are visibly identifiable.

In [ ]:
# Kaggle setup: clone the project when the notebook was imported without repository files.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/ThatTryHard/indonesian-food-vlm-analyzer.git"
REPOSITORY_REVISION = os.environ.get("FOOD_VLM_REVISION", "main")
PROJECT_ROOT = Path(os.environ.get("FOOD_VLM_PROJECT_ROOT", "/kaggle/working/indonesian-food-vlm-analyzer"))

if not (PROJECT_ROOT / "src").exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPOSITORY_REVISION,
        REPOSITORY_URL, str(PROJECT_ROOT),
    ], check=True)

# Notebook 01 only needs packages already supplied by Kaggle's pinned base image.
# Replacing NumPy or Pandas inside a running kernel can mix incompatible binary modules.
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Environment: Kaggle pinned base packages (no in-kernel replacement)")
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import hashlib
import json
import shutil

import kagglehub
import pandas as pd
from IPython.display import display

from src.artifacts import project_protocol_digest
from src.config import artifact_dir, load_config
from src.data import manifest_digest, sha256_file
from src.ontology import IngredientOntology
from src.vlm import build_visible_prompt

config_path = PROJECT_ROOT / "configs/project.json"
ontology_path = PROJECT_ROOT / "data/ontology/visible_ingredients.json"
config = load_config(config_path)
ontology = IngredientOntology.from_json(ontology_path)
ARTIFACT_ROOT = artifact_dir(config)
BENCHMARK_DIR = ARTIFACT_ROOT / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

# Resume safely from a packet exported by an earlier Kaggle session.
# Copy it into writable storage because /kaggle/input is read-only.
packet_override = os.environ.get("FOOD_VLM_BENCHMARK_PACKET")
packet_source = Path(packet_override) if packet_override else None
if packet_source is None and Path("/kaggle/input").exists():
    attached_manifests = list(Path("/kaggle/input").rglob("benchmark_manifest.csv"))
    attached_archives = list(Path("/kaggle/input").rglob("benchmark_packet.zip"))
    candidates = attached_manifests + attached_archives
    if len(candidates) == 1:
        packet_source = candidates[0]
    elif len(candidates) > 1:
        print("Multiple prior packets found; set FOOD_VLM_BENCHMARK_PACKET to resume one explicitly.")
if packet_source is not None and not (BENCHMARK_DIR / "benchmark_manifest.csv").exists():
    if not packet_source.exists():
        raise FileNotFoundError(packet_source)
    if packet_source.is_file() and packet_source.suffix.lower() == ".zip":
        shutil.unpack_archive(str(packet_source), str(BENCHMARK_DIR))
    else:
        source_dir = packet_source.parent if packet_source.is_file() else packet_source
        packet_filenames = {
            "benchmark_manifest.csv", "manifest_lock.json", "image_inventory.csv", "corrupt_images.csv",
            "annotations_annotator_a.csv", "annotations_annotator_b.csv",
        }
        for filename in packet_filenames:
            source_file = source_dir / filename
            if source_file.is_file():
                shutil.copy2(source_file, BENCHMARK_DIR / filename)
    print("Restored writable benchmark packet from:", packet_source)

ontology_table = pd.DataFrame([
    {"id": label.id, "category": label.category, "visual_rule": label.hint}
    for label in ontology.labels
])
print("Primary task:", config["project"]["primary_task"])
print("Ontology version:", ontology.version, "| labels:", len(ontology.ids))
display(ontology_table)

## Validate images and seal the split

Every image is verified, hashed with SHA-256, assigned to a perceptual duplicate group, and sampled once. The manifest contains 20 unique images per class: 12 train, 4 validation, and 4 test.

Split membership is frozen before annotation to prevent duplicate contamination and keep finished labels from influencing test composition. The benchmark uses pre-annotation class balance instead of rearranging samples after ingredient labels are known.

In [ ]:
# Resolve the exact Kaggle dataset by its frozen slug; an explicit environment path can override it.
dataset_config = config["datasets"]["indonesian_target"]
explicit_target = os.environ.get(dataset_config["path_env"])
TARGET_DATASET_ROOT = Path(explicit_target) if explicit_target else Path(kagglehub.dataset_download(dataset_config["slug"]))

print("Dataset slug:", dataset_config["slug"])
print("Resolved root:", TARGET_DATASET_ROOT)
if not TARGET_DATASET_ROOT.exists():
    raise FileNotFoundError(TARGET_DATASET_ROOT)

In [ ]:
# Build once. Existing manifests are never silently overwritten.
manifest_path = BENCHMARK_DIR / "benchmark_manifest.csv"
if not manifest_path.exists():
    subprocess.run([
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_benchmark.py"),
        "--dataset-root", str(TARGET_DATASET_ROOT),
        "--output-dir", str(BENCHMARK_DIR),
    ], check=True)
else:
    print("Using existing sealed manifest:", manifest_path)

manifest = pd.read_csv(manifest_path, keep_default_na=False)
lock = json.loads((BENCHMARK_DIR / "manifest_lock.json").read_text(encoding="utf-8"))

assert len(manifest) == 260, f"Expected 260 rows, found {len(manifest)}"
assert manifest["sample_id"].is_unique
assert manifest.groupby("food_class").size().eq(20).all()
assert sorted(manifest["food_class"].unique()) == sorted(config["benchmark"]["expected_classes"])
assert manifest_digest(manifest) == lock["manifest_sha256"], "Existing manifest no longer matches its lock"
assert sha256_file(ontology_path) == lock["ontology_sha256"], "Ontology changed after benchmark sealing"
assert sha256_file(config_path) == lock["config_sha256"], "Config changed after benchmark sealing"
assert hashlib.sha256(build_visible_prompt(ontology).encode("utf-8")).hexdigest() == lock["vlm_prompt_sha256"]
assert project_protocol_digest(PROJECT_ROOT) == lock["project_protocol_sha256"], "Protocol code changed after sealing"
per_class_splits = manifest.groupby(["food_class", "split"]).size().unstack(fill_value=0)
assert per_class_splits["train"].eq(12).all()
assert per_class_splits["validation"].eq(4).all()
assert per_class_splits["test"].eq(4).all()

print(json.dumps(lock, indent=2, sort_keys=True))
display(per_class_splits)
display(pd.read_csv(BENCHMARK_DIR / "corrupt_images.csv").head())

## Create two independent per-image annotation passes

Each image receives `visible_ingredients`, `uncertain_ingredients`, explicit exclusion or all-negative flags, and notes from two independent passes. The two sheets use different random orders. Per-image judgments are required because a class-level recipe list cannot serve as ground truth for every photograph.

Read `docs/ANNOTATION_GUIDE.md` completely before starting. Annotators must not inspect each other's sheets. If you personally perform both passes, use a washout period and disclose that it is a weaker design than two people.

In [ ]:
print((PROJECT_ROOT / "docs/ANNOTATION_GUIDE.md").read_text(encoding="utf-8"))

In [ ]:
# Choose exactly one sheet for this independent pass, then run this cell.
# Use a fresh session and choose annotator_b for the second pass.
ANNOTATOR_ID = "annotator_a"  # change to annotator_b only in the separate second pass

if ANNOTATOR_ID not in {"annotator_a", "annotator_b"}:
    raise ValueError("ANNOTATOR_ID must be annotator_a or annotator_b")

annotation_path = BENCHMARK_DIR / f"annotations_{ANNOTATOR_ID}.csv"
sheet = pd.read_csv(annotation_path, keep_default_na=False)
from src.annotations import validate_annotation_sheet
sheet = validate_annotation_sheet(sheet, manifest, ontology, require_complete=False)
print("Editing:", annotation_path)
print("Do not open the other annotator's CSV during this pass.")

In [ ]:
from src.annotation_ui import AnnotationApp

app = AnnotationApp(
    sheet=sheet,
    ontology=ontology,
    image_root=TARGET_DATASET_ROOT,
    output_csv=annotation_path,
)
app.display()

## Annotation completion check and export

The validator below does not accept blank rows, unknown labels, or labels marked both visible and uncertain. If an assessable image supports none of the 43 labels, select `No supported ontology label`; never force a positive. Export the packet after each session; `/kaggle/working` is temporary.

In [ ]:
from src.annotations import annotation_progress, validate_annotation_sheet

saved_sheet = pd.read_csv(annotation_path, keep_default_na=False)
progress = annotation_progress(saved_sheet)
print(progress)

if progress["remaining"] == 0:
    validate_annotation_sheet(saved_sheet, manifest, ontology, require_complete=True)
    print("This annotation pass is structurally complete.")
else:
    print("Resume the interface before moving to notebook 02.")

archive_base = ARTIFACT_ROOT / "benchmark_packet"
archive_path = shutil.make_archive(str(archive_base), "zip", BENCHMARK_DIR)
print("Download and preserve:", archive_path)

## Handoff to Notebook 02

Proceed only when `annotations_annotator_a.csv` and `annotations_annotator_b.csv` are independently complete. Upload the benchmark packet as a private Kaggle dataset or attach it to Notebook 02. Notebook 02 validates the lock digest, measures agreement, forces explicit adjudication, and only then trains models.